# 06 — Enrichissement fonctionnel par communauté

**Objectif :** dans le notebook 03, on a interprété les 2 communautés détectées uniquement
en lisant manuellement les fonctions UniProt de leurs membres. Ici, on va plus loin : on
relance une requête d'enrichissement fonctionnel **séparément pour chaque communauté**,
pour obtenir une preuve statistique de la fonction biologique dominante de chaque groupe
(au lieu d'une interprétation qualitative/manuelle).

**Données utilisées :** `data/processed/community_assignments.csv` (notebook 03)

In [1]:
import requests
import pandas as pd
from io import StringIO
import time

community_df = pd.read_csv("../data/processed/community_assignments.csv")
community_df

,cytokine,community_id,degree_centrality,betweenness_centrality,eigenvector_centrality
0,IL6,0,0.90,0.005263,0.264033
1,IL4,0,0.90,0.026316,0.260781
2,TNF,0,0.90,0.115789,0.255677
3,IL1B,0,0.85,0.000000,0.260225
4,IL13,0,0.85,0.026316,0.250273
5,CXCL8,0,0.85,0.021053,0.251777
6,CCL2,0,0.80,0.021053,0.236730
7,IL2,0,0.80,0.010526,0.246352
8,IL5,0,0.75,0.000000,0.236560
9,IL9,0,0.75,0.010526,0.224618


In [2]:
def get_string_functional_enrichment(gene_list, species=9606):
    """Identique à la fonction du script 01, réutilisée ici par sous-groupe de cytokines."""
    string_api_url = "https://string-db.org/api"
    params = {
        "identifiers": "%0d".join(gene_list),
        "species": species,
        "caller_identity": "portfolio_project_cytokines",
    }
    url = f"{string_api_url}/tsv/enrichment"
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    return pd.read_csv(StringIO(response.text), sep="\t")

## Enrichissement pour chaque communauté, séparément

In [3]:
SPECIES_TAXID = 9606
community_enrichments = {}

for community_id in sorted(community_df["community_id"].unique()):
    members = community_df[community_df["community_id"] == community_id]["cytokine"].tolist()
    print(f"Communauté {community_id} ({len(members)} cytokines) : {members}")

    if len(members) < 3:
        print("  -> Moins de 3 membres, enrichissement peu fiable statistiquement, ignoré.\n")
        continue

    enrichment = get_string_functional_enrichment(members, species=SPECIES_TAXID)
    community_enrichments[community_id] = enrichment

    enrichment.to_csv(f"../data/processed/enrichment_community_{community_id}.csv", index=False)
    print(f"  -> {len(enrichment)} termes trouvés, sauvegardé.\n")
    time.sleep(1)

Communauté 0 (14 cytokines) : ['IL6', 'IL4', 'TNF', 'IL1B', 'IL13', 'CXCL8', 'CCL2', 'IL2', 'IL5', 'IL9', 'CCL11', 'CXCL10', 'TGFB1', 'IL1RN']
  -> 926 termes trouvés, sauvegardé.

Communauté 1 (7 cytokines) : ['IL10', 'IFNG', 'IL17A', 'IL22', 'IL23A', 'IL12B', 'IL12A']
  -> 357 termes trouvés, sauvegardé.



## Top 10 termes les plus significatifs, pour chaque communauté

In [4]:
for community_id, enrichment in community_enrichments.items():
    print(f"\n{'='*70}")
    print(f"COMMUNAUTÉ {community_id} — TOP 10 TERMES ENRICHIS")
    print("="*70)

    fdr_col = "fdr" if "fdr" in enrichment.columns else enrichment.columns[-1]

    # On privilégie les catégories les plus interprétables pour un rapport
    interesting_categories = ["Process", "Function", "KEGG", "WikiPathways", "Component"]
    filtered = enrichment[enrichment["category"].isin(interesting_categories)]

    top10 = filtered.sort_values(fdr_col).head(10)
    display(top10[["category", "description", "number_of_genes", fdr_col]])


COMMUNAUTÉ 0 — TOP 10 TERMES ENRICHIS


,category,description,number_of_genes,fdr
816,WikiPathways,Overview of proinflammatory and profibrotic me...,14,3.330000e-28
627,KEGG,Cytokine-cytokine receptor interaction,14,7.320000e-24
408,Function,Cytokine activity,14,8.480000e-24
409,Function,Cytokine receptor binding,14,2.970000e-23
817,WikiPathways,Immune infiltration in pancreatic cancer,10,1.040000e-21
818,WikiPathways,Lung fibrosis,10,6.570000e-20
628,KEGG,IL-17 signaling pathway,10,1.300000e-18
819,WikiPathways,Cytokines and inflammatory response,8,1.870000e-17
820,WikiPathways,Network map of SARS-CoV-2 signaling pathway,11,2.250000e-17
821,WikiPathways,COVID-19 adverse outcome pathway,7,3.310000e-16



COMMUNAUTÉ 1 — TOP 10 TERMES ENRICHIS


,category,description,number_of_genes,fdr
192,KEGG,Inflammatory bowel disease,7,1.140000e-15
328,WikiPathways,Development and heterogeneity of the ILC family,6,1.850000e-13
329,WikiPathways,Overview of proinflammatory and profibrotic me...,7,2.120000e-13
330,WikiPathways,Immune infiltration in pancreatic cancer,6,2.120000e-13
331,WikiPathways,Allograft rejection,6,1.360000e-11
193,KEGG,Cytokine-cytokine receptor interaction,7,2.280000e-11
127,Function,Cytokine activity,7,1.850000e-10
332,WikiPathways,Inflammatory bowel disease signaling,5,2.260000e-10
194,KEGG,JAK-STAT signaling pathway,6,2.360000e-10
333,WikiPathways,T-cell activation SARS-CoV-2,5,5.700000e-09


## Comparaison synthétique des deux communautés

On extrait, pour chaque communauté, le terme fonctionnel (catégorie "Process" ou "Function")
le plus significatif — celui qui résume le mieux la fonction biologique dominante du groupe.

In [5]:
summary_rows = []
for community_id, enrichment in community_enrichments.items():
    fdr_col = "fdr" if "fdr" in enrichment.columns else enrichment.columns[-1]
    process_terms = enrichment[enrichment["category"].isin(["Process", "Function"])]
    if not process_terms.empty:
        top_term = process_terms.sort_values(fdr_col).iloc[0]
        summary_rows.append({
            "community_id": community_id,
            "n_members": len(community_df[community_df["community_id"] == community_id]),
            "top_functional_term": top_term["description"],
            "fdr": top_term[fdr_col],
        })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv("../data/processed/community_functional_summary.csv", index=False)
summary_df

,community_id,n_members,top_functional_term,fdr
0,0,14,Cytokine activity,8.480000e-24
1,1,7,Cytokine activity,1.850000e-10


## Résumé de ce notebook

- Relancé un enrichissement fonctionnel STRING séparément pour chaque communauté
- Identifié le terme biologique le plus significatif par communauté (preuve statistique,
  pas juste une lecture manuelle des fonctions individuelles comme au notebook 03)
- Sauvegardé les résultats détaillés par communauté (`data/processed/enrichment_community_X.csv`)
  et une synthèse comparative (`data/processed/community_functional_summary.csv`)

**Phrase à réutiliser dans le README :**
*"L'enrichissement fonctionnel de chaque communauté confirme statistiquement (test
hypergéométrique corrigé, STRING) que le groupe 1 est associé à [terme], contre [terme]
pour le groupe 0, validant ainsi l'interprétation biologique de la structure détectée par Louvain."*

**Prochaine étape (notebook 07, optionnel) :** prédiction de liens manquants (link prediction).